User's portfolio - Business Problem

A simple way to identify your potential employees based on the business problem you have along with their portfolio - you might hire them for part-time or as a freelancer just for the business problem alone and hire them full-time based on their actual performance - The choice is yours

In [2]:
# All import statements along with groq ai initialization and configuration

import os
import json
import requests
import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display, update_display
from bs4 import BeautifulSoup
load_dotenv()
GROQ_BASE_URL = os.getenv("GROQ_BASE_URL")
GROQ_API_KEY = os.getenv("GROQAI_API_KEY")

In [3]:
# Scrapping the portfolio to analyze the user capabilities
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}
# Scrapes portfolio website content and returns extracted text truncated to 5500 characters
def fetch_website_contents(url):
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    return (title + "\n\n" + text)[:5_500]

In [4]:
# System and User Prompt
system_prompt = """ 
You are a strict senior developer who hires canditates based on their portfolio
"""

# Injecting the user portfolio with business problem for user prompt
# Builds the user evaluation prompt combining scraped portfolio text and business problem description
def get_user_prompt(portfolio, business_problem):
    portfolio_content = fetch_website_contents(portfolio)
    user_prompt = f""" 
    Here is the listed portfolio contents {portfolio_content} for the url {portfolio} and here is the business problem {business_problem}
    I need to select the canditates for the business problem based on their experience, projects. Have a clear understanding on what the user does clearly. 
    Then select the pros and cons of the portfolio with respect to the busiess problem and map them accordingly - how well they suited to the business problems.
    Include the canditate name, role, positives and negatives , worth_hiring in a neat markdown file
    Keep the messages a bit more descriptive so that it will be easier for the clients to take decisions for the people who are hiring for their business needs
    """
    return user_prompt

In [5]:
# Function to compare user's portfolio with business problem
groq = OpenAI(base_url = GROQ_BASE_URL, api_key = GROQ_API_KEY)
# Evaluates candidate portfolio against business problem using Groq API and returns Markdown assessment report
def assess_user(portfolio, business_problem):
    response = groq.chat.completions.create(
        model = os.getenv("GROQ_MODEL"),
        messages = [
            {"role": "system", "content":system_prompt},
            {"role":"user", "content":get_user_prompt(portfolio, business_problem)}
        ]
    )

    result = response.choices[0].message.content
    return result

In [ ]:
# Business problem and portfolio - calling the main function

business_problem = """ 
Client Context: A mid-sized retail brand sells products simultaneously across three channels: their primary web application, a mobile app, and a third-party marketplace.

Core Business Problem - Customers frequently purchase out-of-stock items during peak shopping periods, forcing the business to cancel orders post-checkout and issue refunds.

Technical Root Cause: 
Batch Processing Latency - Inventory levels sync across channels using a scheduled batch job that runs every 4 hours rather than using real-time, event-driven updates.
Siloed Data Stores - The point-of-sale (POS) system and the online e-commerce platform maintain separate databases without a centralized distributed locking mechanism or single source of truth during checkout.

Business Impact:
Lost Revenue - 14% order cancellation rate during promotional campaigns, leading to direct refund processing fees and lost sales.
Operational Overhead - Inbound support ticket volume increased by 35%, primarily driven by angry customers inquiring about canceled orders.
Platform Penalties - The third-party marketplace lowered the seller rating due to high fulfillment defect rates, reducing product search visibility. 

"""
# user_report = assess_user("https://shraghav.github.io/", business_problem )
view = gr.Interface(assess_user )
